# **Clustering des films**

### **Imports**

In [16]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pymongo import MongoClient

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [2]:
load_dotenv()
db = MongoClient(os.getenv("MONGO_URI"))["cinema"]

In [3]:
films = pd.DataFrame(db["films"].find(
    {"budget": {"$ne": None}, "recettes": {"$ne": None}},          # films complets uniquement
    {"titre": 1, "annee": 1, "budget": 1, "recettes": 1,
     "note_moyenne": 1, "nb_votes": 1, "duree_min": 1, "genres": 1}
))

print(len(films), "films avec budget et recettes connus")
films.describe()

4198 films avec budget et recettes connus


,_id,duree_min,budget,recettes,note_moyenne,nb_votes,annee
count,4.198000e+03,4198.000000,4.198000e+03,4.198000e+03,4198.000000,4198.000000,4198.000000
mean,2.322988e+05,109.691758,4.484062e+07,1.303798e+08,6.597773,3621.340638,2012.210100
std,2.902351e+05,18.831097,5.333607e+07,2.246055e+08,0.740933,4444.237714,6.795293
min,1.200000e+01,57.000000,1.000000e+00,6.000000e+00,2.939000,500.000000,2000.000000
25%,1.093725e+04,96.000000,1.050000e+07,1.537284e+07,6.119000,1003.000000,2007.000000
50%,8.027600e+04,106.000000,2.500000e+07,5.154869e+07,6.600000,1957.500000,2012.000000
75%,3.927720e+05,120.000000,6.000000e+07,1.417588e+08,7.113000,4286.000000,2017.000000
max,1.472638e+06,275.000000,6.588000e+08,2.942661e+09,8.800000,41999.000000,2025.000000


Problèmes des valeurs impossibles : 

- Budget MIN = 1($)
- Recettes MiN = 6($)
Aucun film de cinéma à +500 votes ne coûte 1$. Il s'agit sûrement d'erreurs de saisie dans TMDB (base remplie par des bénévoles).
On parle d'ici d'outliers ou valeurs aberrantes qu'il faudra analyser et traîter. 

Egalement la moyenne est plus élevée que la médiane à cause des films blockbusters qui tire la moyenne vers le haut. 

## **Analyses en détail et traitement des valeurs aberrantes**

In [4]:
suspects = films[(films["budget"] < 10_000) | (films["recettes"] < 10_000)]
print(len(suspects), "films avec un budget ou des recettes < 10 000 $")
suspects[["titre", "annee", "budget", "recettes"]].sort_values("budget").head(15)

13 films avec un budget ou des recettes < 10 000 $


,titre,annee,budget,recettes
3892,The quiet girl,2022,1,6
4168,Rampage : Sniper en liberté,2009,120,428
2668,Freaks,2019,2000,368410
1612,Primer,2004,7000,545436
3254,Sniper : Le Corbeau Blanc,2022,1000000,881
3765,Honeymoon,2014,1000000,9318
3200,Blood and Bone,2009,3700000,3492
4113,The Sacrament,2014,4000000,9221
697,High School Musical : Premiers pas sur scène,2006,4200000,3746
3375,Mémoires d’un escargot,2024,4350000,7


**1.Erreurs de saisie**

- *The Quiet Girl* : 1$ de budget et 6$ de recettes
- *Rampage* : 120$ et 428$

**2.Les films des plateformes de streaming ou diffusés à la Tv**

- *La Demoiselle et le Dragon* : 60M$ de budget et 5000$ de recettes -> Diffusé sur Netflix
- *Le Cercle des neiges* : 65.5M$ de budget et 1281$ de recettes -> Diffusé sur Netflix
- *High School Musical* : téléfilm Disney Channel
Budget réel, mais pas ou presque pas de sorties au cinéma. Les recettes proviennent des abonnements aux plateformes. 

💡​ Le film *Primer* est connu pour son budget de 7000$ et sa recette de 545436$. 

Sur nos données totales, ces films représentent une faibles part, on peut aisément les retirer car il perturberait nos modèles de Machine Learning. 

In [5]:
films = films[(films["budget"] >= 10_000) & (films["recettes"] >= 10_000)].copy()
print(len(films), "films conservés")

4185 films conservés


### **Mise à l'échelle des données**

La différence de budget et recette est très **déséquilibrée** : La plupart des films rapportent des millions, et quelques blockbusters rapportent des milliards de $.
Il faut repasser les données à l'échelle logarithmique afin que chaque film puisse "peser" le même poids sans écarts de valeurs. 

In [6]:
films["log_budget"] = np.log10(films["budget"])
films["log_recettes"] = np.log10(films["recettes"])

# Comparaison avant / après
fig = px.histogram(films, x="recettes", nbins=60,
                   title="AVANT : distribution des recettes (en $)")
fig.show()

fig = px.histogram(films, x="log_recettes", nbins=60,
                   title="APRÈS : distribution des recettes (log10)")
fig.show()

Avant : presque tous les films sont tassés dans la première barre à gauche, avec une longue traîne de quelques blockbusters vers la droite. On ne distingue rien.
Après : les films s'étalent en une forme de cloche, centrée autour de 7,7 (cad entre 60 et 100M$). On voit enfin la diversité des films.

### **Standardisation des données**

Les variables ont des échelles très différentes il faut donc les standardiser afin qu'elle puise peser le même poids. 

In [8]:
films["log_votes"] = np.log10(films["nb_votes"])

variables = ["log_budget", "log_recettes", "note_moyenne", "log_votes", "duree_min"]
X = StandardScaler().fit_transform(films[variables])

print("Forme du tableau :", X.shape)                         # (nombre de films, nombre de variables)
print("Moyennes après standardisation :", X.mean(axis=0).round(2))
print("Écarts-types après standardisation :", X.std(axis=0).round(2))

Forme du tableau : (4185, 5)
Moyennes après standardisation : [-0. -0.  0.  0.  0.]
Écarts-types après standardisation : [1. 1. 1. 1. 1.]


In [10]:
resultats = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquettes = kmeans.fit_predict(X)
    resultats.append({
        "k": k,
        "inertie": kmeans.inertia_,
        "silhouette": silhouette_score(X, etiquettes),
    })

resultats = pd.DataFrame(resultats)

px.line(resultats, x="k", y="inertie", markers=True,
        title="Méthode du coude : inertie selon le nombre de groupes").show()
px.line(resultats, x="k", y="silhouette", markers=True,
        title="Score de silhouette selon le nombre de groupes").show()
resultats.round(3)

,k,inertie,silhouette
0,2,14388.391,0.273
1,3,11860.542,0.227
2,4,10403.277,0.223
3,5,9290.502,0.206
4,6,8599.993,0.202
5,7,8057.768,0.188
6,8,7598.938,0.178
7,9,7192.643,0.178
8,10,6868.991,0.179


In [11]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
films["groupe"] = kmeans.fit_predict(X)

portrait = films.groupby("groupe").agg(
    nb_films=("titre", "count"),
    budget_M=("budget", lambda s: s.median() / 1e6),
    recettes_M=("recettes", lambda s: s.median() / 1e6),
    note=("note_moyenne", "median"),
    votes=("nb_votes", "median"),
    duree=("duree_min", "median"),
).round(1)
portrait

,nb_films,budget_M,recettes_M,note,votes,duree
groupe,,,,,,
0,1458,30.0,62.2,6.1,1713.5,99.0
1,979,90.0,273.3,7.0,7479.0,120.0
2,748,5.0,3.8,6.5,874.5,97.0
3,1000,17.1,32.2,7.1,1681.0,118.0


In [12]:
for g in range(4):
    exemples = films[films["groupe"] == g].sort_values("nb_votes", ascending=False)["titre"].head(6)
    print(f"Groupe {g} :", ", ".join(exemples))

Groupe 0 : American Nightmare, Les 4 Fantastiques et le Surfer d'argent, Scary Movie, Cloverfield, Sausage Party, Easy Girl
Groupe 1 : Interstellar, Inception, Avengers, The Dark Knight : Le Chevalier noir, Avatar, Deadpool
Groupe 2 : La Plateforme, It Follows, Enemy, Mister Babadook, Paranormal Activity, [REC]
Groupe 3 : Whiplash, Donnie Darko, Call Me by Your Name, American Psycho, Requiem for a Dream, Saw


**Interprétation des groupes**
| Groupe | Nom | Profil |
|---|---|---|
| 1 | Blockbusters | très gros budget et recettes, très vus, bien notés |
| 3 | Films d'auteur acclamés | budget modeste, meilleure note, films longs |
| 0 | Divertissement grand public | budget moyen, note la plus basse, films courts |
| 2 | Petits films à diffusion limitée | très petit budget, peu vus, recettes souvent inférieures au budget |




- L'argent ne fait pas la qualité. Le groupe 3 (films d'auteur) a un budget 5 fois plus faible que les blockbusters, mais une meilleure note.
- Groupes 0 et 3 : même budget, destins opposés. Leurs budgets sont proches (30 M$ contre 17 M$), mais leurs notes s'opposent (6,1 contre 7,1). 
Ce qui les sépare, c'est la qualité perçue et la durée, pas l'argent.


### *ACP*

In [19]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X)                       # chaque film devient un point en 2D
films["axe_1"] = coords[:, 0]
films["axe_2"] = coords[:, 1]

centres = pca.transform(kmeans.cluster_centers_)    # les centres des groupes, projetés en 2D
info = pca.explained_variance_ratio_.sum() * 100
print(f"Information conservée par les 2 axes : {info:.0f} %")

fig = px.scatter(
    films, x="axe_1", y="axe_2",
    color="famille", color_discrete_map=couleurs,
    hover_name="titre", opacity=0.5,
    title=f"Les 4 familles de films (K-means, projection ACP : {info:.0f} % de l'information)",
    labels={"axe_1": "Axe 1", "axe_2": "Axe 2", "famille": "Famille"},
)

# Ajout des centres de groupes (grosses croix noires avec leur nom)
for g, (x, y) in enumerate(centres):
    fig.add_scatter(x=[x], y=[y], mode="markers",
                    marker=dict(symbol="x", size=18, color="black"),
                    showlegend=False)
    fig.add_annotation(x=x, y=y, text=f"<b>{noms[g]}</b>", showarrow=False, yshift=22,
                       bgcolor="white", bordercolor="black", borderwidth=1)

fig.update_layout(height=650)
fig.show()

Information conservée par les 2 axes : 73 %


In [21]:
from pymongo import UpdateOne

maj = [UpdateOne({"_id": row["_id"]}, {"$set": {"famille": row["famille"]}})
       for _, row in films.iterrows()]
resultat = db["films"].bulk_write(maj)
print(resultat.modified_count, "films mis à jour")

4185 films mis à jour
